# Chatbot with tools

In [1]:
from dotenv import load_dotenv
import os

## Setup API Keys

In [2]:
load_dotenv()
SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")

assert SARVAM_API_KEY is not None, "Could NOT load SARVAM_API_KEY from .env"

## Instantiate the Sarvam AI Client

In [3]:
from sarvamai import SarvamAI

In [4]:
client = SarvamAI(
    api_subscription_key=SARVAM_API_KEY,
)

## Use Case Definition

Make the chatbot interact with customers, online helping them:

1. Check the Account Balance
2. Review the list of transactions in their account in the last 24 hours

## Setup the tools

In [5]:
def get_balance(account_number: str):
    if account_number == "001002":
        return "INR 51250"
    else:
        return "INR 4300"

def get_transactions(account_number: str):
    if account_number == "001002":
        return [
            ("Time", "Transaction", "INR"),
            ("11:21 AM", "Debit - XYZ Super Market", "INR 651.50"),
            ("4:45 PM",  "Credit - Refund from PQR Braodband Services", "INR 999"),
        ]
    else:
        return "No transactions in the last 24 hrs in your account"

In [9]:
get_transactions("001002")

[('Time', 'Transaction', 'INR'),
 ('11:21 AM', 'Debit - XYZ Super Market', 'INR 651.50'),
 ('4:45 PM', 'Credit - Refund from PQR Braodband Services', 'INR 999')]

## Definition of tools

In [10]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_balance",
            "description": "Get the balance for the account number",
            "parameters": {
                "type": "object",
                "properties": {
                    "account_number": {"type": "string", "description": "Account Number"},
                },
                "required": ["account_number"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_transactions",
            "description": "Get the transactions for the account number",
            "parameters": {
                "type": "object",
                "properties": {
                    "account_number": {"type": "string", "description": "Account Number"},
                },
                "required": ["account_number"],
            },
        },
    },
]

## chat() to call Sarvam API

In [30]:
def chat(messages: list, tools: list):
    while True:
        response = client.chat.completions(
            model="sarvam-105b-conversations",
            messages=messages,
            tools=tools
        )

        if response.choices[0].finish_reason == "stop":
            break;
        
        # 1. Read list of tool_calls requested by AI and append to chat_history
        tool_calls = response.choices[0].message.tool_calls
        messages.append(
            {
                "role": "assistant",
                "tool_calls": tool_calls
            }
        )
    
        handle_tool_calls(tool_calls, messages)
        
    # 3. Call AI again with updated history to process further 
    
    return response

In [43]:
import json
def handle_tool_calls(tool_calls: list, messages: list):
    # 2. For each tool
    for tool_call in tool_calls:
    
        # a. Get the function name and arguments
        f_name = tool_call.function.name
        f_args = json.loads(tool_call.function.arguments)
        
        # b. Invoke the function and get the result
        if f_name == "get_balance":
            result = get_balance(**f_args)
        elif f_name == "get_transactions":
            result = get_transactions(**f_args)
        print("Result from Tool Call :: ", result)   
        
        # c. Add the result to chat_history
        messages.append(
            {
                "role": "tool",
                "content": str(result),
                "tool_call_id": tool_call.id
                
            }
        )

In [36]:
messages = [
            {"role": "user", "content": "Get the balance and transactions for account 001002 and 12344"},
        ]

response = chat(messages, tools)
print(response.choices[0].message.content)
#print(response.model_dump_json(indent=2))

Here are the results for both accounts:

**Account 001002**
- **Balance:** INR 51,250
- **Transactions:**
  - 11:21 AM — Debit — XYZ Super Market (INR 651.50)
  - 4:45 PM — Credit — Refund from PQR Broadband Services (INR 999)

**Account 12344**
- **Balance:** INR 4,300
- **Transactions:** No transactions in the last 24 hours.


## Chat with the bot

In [40]:
BLUE_TEXT = "\033[94m"
BLACK_TEXT = "\033[0m"

In [41]:
def run_chatbot():

    # 1. initialize chat history with System prompt
    # 2. Get user message and add it to chat history
    # 3. Call chat() to get response from AI 
    # 4. Print the response and add it to chat history
    # 5. Repeat steps 2-4 until user types "quit"

    # 1. initialize chat history with System prompt
    chat_history=[
        {"role": "system", "content": "You are a Customer Service Rep from ABC Bank."},
    ]

    while True:
        
        # 2. Get user message and add it to chat history
        user_message = input("\nUser:")

        if user_message == "quit":
            break
        
        chat_history.append(
            {"role": "user", "content": user_message.strip()}
        )
    
        # 3. Call chat() to get response from AI 
        response = chat(chat_history, tools)
    
        # 4. Print the response and add it to chat history
        print("\n", BLUE_TEXT, "Assistant: ", response.choices[0].message.content.strip(), BLACK_TEXT)
        
        chat_history.append(
            {"role": "assistant", "content": response.choices[0].message.content}
        )

#### Run the chatbot

In [44]:
run_chatbot()


User: Hi, I am Amit, How are you?



  Assistant:  Hi Amit! I'm doing well, thank you for asking. How can I assist you today with your account at ABC Bank? 



User: get the balance for account 001002


Result from Tool Call ::  INR 51250

  Assistant:  The balance for account **001002** is **INR 51,250**. Is there anything else you need help with today, Amit? 



User: transactions 


...


User: balance for 12344 account


Result from Tool Call ::  INR 4300

  Assistant:  The balance for account **12344** is **INR 4,300**. Let me know if you need anything else! 



User: balance for 001002 account



  Assistant:  The balance for account **001002** is **INR 51,250**. Is there anything else I can help you with today, Amit? 



User: quit
